# Notebook 3: Flood Watch app inside Jupyter

This notebook runs everything the Streamlit app does, **one feature per cell**, with results shown right here.
Cells that say *change this* have a variable at the top: edit it and run the cell again.

Needs: notebook 1 finished (model saved), notebook 2 finished (recorded data), and `app_core.py` in `Flood_Watch\app`.

> Research prototype. Not an official warning. Do not use for emergency decisions.

In [ ]:
# CELL 1: setup
import os, sys, json, warnings
from pathlib import Path
from datetime import datetime
import numpy as np, pandas as pd
from IPython.display import display, Image
warnings.filterwarnings("ignore")

BASE = Path(os.environ.get("FLOOD_BASE", str(Path.cwd())))
APP = Path(os.environ.get("FLOOD_APP", str(BASE / "app")))
sys.path.insert(0, str(APP))
import app_core as core

EXPORT = core.WORK / "exports"; EXPORT.mkdir(parents=True, exist_ok=True)
def show(fig, dpi=110):
    display(Image(data=core.fig_png(fig, dpi)))

print("app folder:", APP, "| app_core.py found:", (APP / "app_core.py").exists())
print("model present:", (core.WORK / "models" / "unet_fp16.pt").exists(), "| dataset folder:", core.DATA.exists())
print("recorded data files:", sorted(p.name for p in core.FIX.glob("*.json")))
print("exports will be saved in:", EXPORT)

In [ ]:
# CELL 2: load the state map, the model and the recorded data date
STATES = core.load_states()
MODEL, DEVICE, META, FI = core.load_model()
print("India states loaded:", len(STATES), "| model device:", DEVICE, "| encoder:", META["encoder"], "| threshold:", round(META["threshold"], 2))
fix = core.load_fix("rain_national.json")
print("Offline demo: live data recorded on", core.rec_datetime(fix).strftime("%d %b %Y at %H:%M"), "IST")

## Feature 1: satellite flood mapping on a sample chip

In [ ]:
# CELL 3: list the sample chips (official test split, chips with visible water; India first)
SAMPLES = core.sample_chip_list()
display(pd.DataFrame([{"index": i, "chip": s["chip"], "water in hand label (%)": round(100 * s["water_share"], 1), "India": s["india"]}
                      for i, s in enumerate(SAMPLES)]))

In [ ]:
# CELL 4: analyse one sample chip.  <- change CHIP_INDEX and run again
CHIP_INDEX = 0
chip = SAMPLES[CHIP_INDEX]["chip"]
res = core.load_sample(chip, MODEL, DEVICE, META, FI)
s = res["stats"]
print(f"{chip}: water covers {s['percent_water']:.1f}% of the image ({s['water_km2']:.2f} km2) | bucket: {FI.severity(s['percent_water'])} | agreement with hand label (IoU): {res['iou']:.2f}")
show(core.overlay_fig(res["display"], res["mask"], "Model: water", res["label"].astype(float), "Hand label"))
if "permanent" in res:
    new = res["mask"] & ~res["permanent"]
    print(f"Water not in the permanent-water layer (rough 'newly flooded'): {100 * new.sum() / max(s['valid_pixels'], 1):.1f}% of the image")
print("This is the percent of the image covered by water. It is not a flood probability.")

## Feature 2: analyse your own Sentinel-1 GeoTIFF

In [ ]:
# CELL 5: analyse an uploaded-style file.  <- change UPLOAD_PATH (a georeferenced .tif with VV and VH bands)
UPLOAD_PATH = os.environ.get("FLOOD_UPLOAD", str(core.DATA / "data" / "S1GRDHand" / (SAMPLES[0]["chip"] + "_S1Hand.tif")))
p = Path(UPLOAD_PATH)
r = core.analyse_bytes(p.read_bytes(), p.name, MODEL, DEVICE, META, FI)
s = r["stats"]
print(f"{p.name}: {s['percent_water']:.1f}% water, {s['water_km2']:.2f} km2, bucket: {FI.severity(s['percent_water'])} ({s['note']})")
fig = core.overlay_fig(r["display"], r["mask"], "Model: water")
show(fig)
(EXPORT / "upload_overlay.png").write_bytes(core.fig_png(fig))
(EXPORT / "upload_water_mask.tif").write_bytes(core.mask_geotiff_bytes(r["mask"], r["transform"], r["crs"]))
print("saved overlay and mask to:", EXPORT)

In [ ]:
# CELL 6: before and after.  <- fill in two files of the SAME area on the SAME pixel grid; leave empty to skip
BEFORE_PATH = os.environ.get("FLOOD_BEFORE", "")
AFTER_PATH = os.environ.get("FLOOD_AFTER", "")
if BEFORE_PATH and AFTER_PATH:
    rb = core.analyse_bytes(Path(BEFORE_PATH).read_bytes(), Path(BEFORE_PATH).name, MODEL, DEVICE, META, FI)
    ra = core.analyse_bytes(Path(AFTER_PATH).read_bytes(), Path(AFTER_PATH).name, MODEL, DEVICE, META, FI)
    cmp_ = core.compare_pair(rb, ra)
    print(f"water before {cmp_['percent_before']:.1f}% | water after {cmp_['percent_after']:.1f}% | newly flooded {cmp_['percent_new']:.1f}% ({cmp_['km2_new']:.2f} km2)")
    show(core.overlay_fig(ra["display"], cmp_["mask"], "Newly flooded water (after minus before)"))
else:
    print("Skipped: set BEFORE_PATH and AFTER_PATH to compare two images.")

## Feature 3: live watch (forecast heavy-rain exposure by state)

In [ ]:
# CELL 7: state ranking.  <- change HORIZON to "24h", "48h" or "72h"
HORIZON = "72h"
rank, rec = core.national_ranking()
col = f"{HORIZON} share of points >= 64.5 mm (%)"
top = rank.sort_values([col, f"{HORIZON} highest (mm)"], ascending=False)
display(top[["State", "Points", col, f"{HORIZON} highest (mm)"]].head(15))
from matplotlib.figure import Figure
fig = Figure(figsize=(7, 4)); ax = fig.subplots()
t = top.head(12).iloc[::-1]
ax.barh(t["State"], t[col], color="#1b6ca8"); ax.set_xlabel("Share of monitored points at or above 64.5 mm (%)")
ax.set_title(f"Forecast heavy-rain exposure, {HORIZON} (recorded data, not a flood probability)", fontsize=9); fig.tight_layout()
show(fig)

## Feature 4: state report

In [ ]:
# CELL 8: state situation.  <- change STATE (any India state name; Assam, Kerala, Bihar, Maharashtra, West Bengal have full detail)
STATE = "Assam"
ctx = core.state_context(STATE, STATES)
if ctx.get("empty"):
    print("No recorded data for", STATE)
else:
    print(ctx["summary"], "\n")
    display(ctx["horizons"])
    show(core.forecast_fig(ctx)); show(core.state_map_fig(ctx, STATES))
    if ctx.get("months") is not None:
        display(ctx["months"]); show(core.rain_history_fig(ctx))
        if len(ctx["heavy_list"]):
            print("Days with heavy-rainfall-level values (rain days, not floods):"); display(ctx["heavy_list"])
    else:
        print("Detailed history is recorded only for the pilot states.")
    if ctx.get("rivers") is not None and len(ctx["rivers"]):
        display(ctx["rivers"])
        show(core.river_fig(ctx, ctx["rivers"].iloc[0]["Point"]))
    print("Reported flood events (GDACS, last 90 days):")
    if ctx["events"]:
        display(core.events_frame(ctx["events"]))
    else:
        print("  No large flood event found in the GDACS list for this state. Absence does not prove no flooding.")
    if len(ctx["neighbours"]):
        print("Neighbouring states:"); display(ctx["neighbours"])

In [ ]:
# CELL 9: save the state report as a PDF (uses the STATE from Cell 8)
pdf = core.build_state_pdf(ctx, STATES)
out = EXPORT / f"StateReport_{STATE.replace(' ', '_')}_{datetime.now():%Y%m%d}.pdf"
out.write_bytes(pdf)
print("report saved:", out, f"({len(pdf) / 1e3:.0f} KB)")

## Feature 5: advisory outbox (rule-based, offline; nothing is sent over the internet)

In [ ]:
# CELL 10: list advisory candidates.  <- change SIMULATE to None, or a pilot state name to add an artificial, clearly marked heavy-rain event
SIMULATE = "Assam"
cands = core.make_candidates(SIMULATE)
if not cands:
    print("No candidates under the rules for the recorded data. Set SIMULATE = 'Assam' to demonstrate the alert path.")
for i, c in enumerate(cands):
    print(f"[{i}] {c['state']} - {c['level']}" + ("  (SIMULATION)" if c["simulated"] else "") + ("  [cooldown active]" if c["cooldown"] else ""))
    print(c["message"]); print("-" * 60)

In [ ]:
# CELL 11: human review.  <- set APPROVE_INDEX to a candidate number to approve it (writes a file to the outbox), or REJECT_INDEX to reject it
APPROVE_INDEX = None
REJECT_INDEX = None
if APPROVE_INDEX is not None:
    c = cands[APPROVE_INDEX]
    if c["cooldown"]:
        print("Cooldown is active for", c["state"], "- not written.")
    else:
        print("approved ->", core.record_decision(c, "approved"))
if REJECT_INDEX is not None:
    core.record_decision(cands[REJECT_INDEX], "rejected"); print("rejected and logged")
log = core.ALERTS / "log.csv"
if log.exists():
    display(pd.read_csv(log).tail(10))
    print("outbox folder:", core.ALERTS / "outbox", "| files:", [f.name for f in sorted((core.ALERTS / "outbox").glob("*.txt"))][-5:])

## Optional: open the full Streamlit app from here

In [ ]:
# CELL 12: start the Streamlit app in your browser (set LAUNCH = True). Stop it later with: proc.terminate()
import subprocess
LAUNCH = False
if LAUNCH:
    proc = subprocess.Popen([sys.executable, "-m", "streamlit", "run", str(APP / "streamlit_app.py")], cwd=str(APP))
    print("Streamlit is starting; your browser should open at http://localhost:8501")
else:
    print("Not launched. Set LAUNCH = True and run this cell to start the app.")